# GeomHerd — Exploratory Visualization Notebook
**Paper:** [arXiv:2605.11645](https://arxiv.org/abs/2605.11645)  

This notebook visualizes the geometric evolution of the agent interaction graph during
herding cascades, replicating the spirit of Figures 1–5 in the paper:

- **Figure 1 replica:** κ̄_OR(t) rising before V_a(t) crosses θ_event
- **Figure 2 spirit:** Curvature sign structure during cascade
- **Figure 5a spirit:** V_eff vs κ̄_OR cross-correlation
- **Table 8 spirit:** CUSUM operating-point sweep

All visualizations use synthetic/rule-based data (no LLM, no POT required for most cells).

In [ ]:
# Setup
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for notebook
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from geomherd.simulation.cws_substrate import CWSSubstrate
from geomherd.graph.agent_graph import AgentGraph
from geomherd.geometry.vocabulary import FSQVocabularyTracker
from geomherd.detection.cusum import HerdingDetector, CUSUMDetector
from geomherd.evaluation.metrics import DetectionMetrics
from geomherd.utils.config import set_global_seed

set_global_seed(42)
print("Imports OK. matplotlib backend:", matplotlib.get_backend())

In [ ]:
# Generate a supercritical CWS trajectory
N, T = 30, 300
substrate = CWSSubstrate(N=N, na=2, kappa=1.8, seed=7)
ag = AgentGraph(N=N, Tw=30, w0=0.3, delta_t=5)
vocab = FSQVocabularyTracker()
det = HerdingDetector(operating_point='recall', baseline_window=15, skip_initial=15)

actions = substrate.reset(seed=7)

order_params, veff_series = [], []
proxy_kbar_series, snapshot_ts = [], []
edge_count_series = []
alarm_t = None

for t in range(T):
    actions, prices, info = substrate.step()
    V_eff = vocab.effective_vocab(actions)
    veff_series.append(V_eff)
    order_params.append(info['order_parameter'])
    snapshot_ready = ag.push(actions)
    if snapshot_ready:
        W = ag.get_weight_matrix()
        edges = ag.get_edge_list()
        proxy_kbar = float(W[W > ag.w0].mean()) if W.max() > ag.w0 else 0.0
        proxy_kbar_series.append(proxy_kbar)
        edge_count_series.append(len(edges))
        snapshot_ts.append(t)
        _, alarm = det.update(proxy_kbar)
        if alarm and alarm_t is None:
            alarm_t = t

herding_t = next((t for t, v in enumerate(order_params) if v > 0.5), None)
print(f"Trajectory: N={N}, kappa=1.8, T={T}")
print(f"Herding event (V_a>0.5): step {herding_t}")
print(f"CUSUM alarm (proxy κ̄):  step {alarm_t}")
if alarm_t and herding_t:
    print(f"Lead: {herding_t - alarm_t} steps")

In [ ]:
# === Figure 1 spirit: κ̄_OR(t) rises before V_a(t) ===
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=False)

# Panel (a): Order parameter
ax0 = axes[0]
ax0.plot(range(T), order_params, 'k-', lw=1.2, label='$V_a(t)$ (order parameter)')
ax0.axhline(0.5, color='red', ls='--', lw=1, label='$\\theta_{event}=0.50$')
if herding_t:
    ax0.axvline(herding_t, color='red', ls='-', lw=1.5, alpha=0.5, label=f'$\\tau^\\star$={herding_t}')
ax0.set_ylabel('$V_a(t)$')
ax0.set_title('Panel (c): Order parameter — herding event')
ax0.legend(fontsize=8)
ax0.set_ylim(0, 1)

# Panel (b): Proxy curvature signal (mean edge weight > w0)
ax1 = axes[1]
ax1.plot(snapshot_ts, proxy_kbar_series, 'b-', lw=1.5, label='Proxy $\\bar{\\kappa}_{OR}^+(t)$ (mean w)')
ax1.axhline(0.60, color='blue', ls='--', lw=1, label='$\\theta_{geom}$ (proxy)')
if alarm_t:
    ax1.axvline(alarm_t, color='blue', ls='-', lw=1.5, alpha=0.5, label=f'CUSUM alarm t={alarm_t}')
if herding_t:
    ax1.axvline(herding_t, color='red', ls='--', lw=1.5, alpha=0.4, label=f'$\\tau^\\star$={herding_t}')
ax1.set_ylabel('Proxy $\\bar{\\kappa}^+$')
ax1.set_xlabel('Simulator step $t$')
ax1.set_title('Panel (b): Agent-graph curvature signal (proxy) — alarm fires before event')
ax1.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../results/fig1_spirit_kappa_leads_event.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: results/fig1_spirit_kappa_leads_event.png")
print("Paper Figure 1: geometry leads herding — κ̄_OR rises through θ_geom=0.30 before V_a crosses θ_event=0.50")

In [ ]:
# === Figure 5a spirit: V_eff vs κ̄_OR cross-correlation ===
# Paper: V_eff leads κ̄_OR by ~15 steps (cross-correlation peak at lag ~15)

# Align V_eff series to snapshot timestamps
veff_at_snapshots = np.array([veff_series[t] for t in snapshot_ts])
kbar_arr = np.array(proxy_kbar_series)

# Normalize both series
def normalize(x):
    return (x - x.mean()) / (x.std() + 1e-8)

veff_n = normalize(veff_at_snapshots)
kbar_n = normalize(kbar_arr)

# Cross-correlation: xcorr[lag] = corr(V_eff[t], κ̄[t+lag])
max_lag = min(30, len(veff_n) // 3)
lags = np.arange(-max_lag, max_lag + 1)
xcorr = np.array([
    np.corrcoef(veff_n[:len(veff_n)-abs(lag)],
                kbar_n[abs(lag):] if lag >= 0 else kbar_n[:len(kbar_n)-abs(lag)])[0, 1]
    if abs(lag) < len(veff_n) else 0.0
    for lag in lags
])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(lags, xcorr, 'b-', lw=2)
ax.axvline(0, color='k', ls='--', lw=0.8)
peak_lag = lags[np.argmax(xcorr)]
ax.axvline(peak_lag, color='green', ls='--', lw=1.5,
           label=f'Peak lag = {peak_lag} steps (V_eff leads)')
ax.set_xlabel('Lag (simulator steps, positive = V_eff leads κ̄)')
ax.set_ylabel('Cross-correlation')
ax.set_title('Figure 5a spirit: Agent-graph V_eff vs proxy κ̄_OR cross-correlation\n'
             '(Paper: peak at lag≈15, V_eff leading)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../results/fig5a_spirit_veff_xcorr.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Peak cross-correlation lag: {peak_lag} steps")
print("Paper Figure 5a: peak at lag≈15 with V_eff leading κ̄_OR")

In [ ]:
# === Edge density evolution during cascade ===
# Visualizes graph densification as herding develops (spirit of Figure 2)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: edge count over time
ax = axes[0]
ax.plot(snapshot_ts, edge_count_series, 'purple', lw=1.5)
if alarm_t:
    ax.axvline(alarm_t, color='blue', ls='--', lw=1.5, label=f'Alarm t={alarm_t}')
if herding_t:
    ax.axvline(herding_t, color='red', ls='--', lw=1.5, label=f'Event τ*={herding_t}')
ax.set_xlabel('Simulator step')
ax.set_ylabel('Retained edge count |E_t|')
ax.set_title('Graph densification during herding cascade\n'
             'Paper Fig 2: graph contracts into dense clique')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Right: V_eff over time  
ax2 = axes[1]
ax2.plot(range(T), veff_series, 'orange', lw=1.5, label='$V_{eff}(t)$')
ax2.plot(snapshot_ts, proxy_kbar_series * 5, 'b--', lw=1.2, alpha=0.7,
         label='Proxy κ̄ (×5 for scale)')
if alarm_t:
    ax2.axvline(alarm_t, color='blue', ls=':', lw=1.5, label=f'Alarm t={alarm_t}')
if herding_t:
    ax2.axvline(herding_t, color='red', ls=':', lw=1.5, label=f'Event τ*={herding_t}')
ax2.set_xlabel('Simulator step')
ax2.set_ylabel('$V_{eff}$ (behavioral diversity)')
ax2.set_title('Behavioral homogenization: V_eff contracts\n'
              'Paper: V_eff leads κ̄_OR by ~15 steps')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../results/fig2_spirit_edge_density.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: results/fig2_spirit_edge_density.png")

In [ ]:
# === Table 8 spirit: CUSUM operating-point sweep ===
# Reproduce the recall vs FAR tradeoff across (k_sigma, h_sigma)

rng = np.random.default_rng(0)
N_traj = 60  # 30 super, 30 sub
T_sweep = 250

def synthetic_kbar(supercritical, seed, T=T_sweep):
    rng_t = np.random.default_rng(seed)
    if supercritical:
        onset = rng_t.integers(80, 140)
        event_t = onset + rng_t.integers(15, 30)
        series = np.concatenate([
            rng_t.normal(0.10, 0.015, onset),
            rng_t.normal(0.42, 0.03, T - onset)
        ])
    else:
        series = rng_t.normal(0.10, 0.015, T)
        event_t = None
    return series, event_t

k_sigmas = [0.25, 0.5, 1.0, 2.0]
h_sigmas = [3.0, 4.0, 6.0]

results_grid = {}
for k_sig in k_sigmas:
    for h_sig in h_sigmas:
        alarm_ts_list, event_ts_list, is_sup_list = [], [], []
        for i in range(N_traj):
            sup = i < N_traj // 2
            series, event_t = synthetic_kbar(sup, seed=i)
            det = CUSUMDetector(k=k_sig * 0.025, h=h_sig * 0.025, baseline_window=25)
            alarm_t = None
            for t, v in enumerate(series):
                _, alarm = det.update(v)
                if alarm and alarm_t is None:
                    alarm_t = t
                    break
            alarm_ts_list.append(alarm_t)
            event_ts_list.append(event_t)
            is_sup_list.append(sup)
        prf = DetectionMetrics.precision_recall_far(alarm_ts_list, event_ts_list, is_sup_list)
        lead = DetectionMetrics.conditional_lead_time(alarm_ts_list, event_ts_list)
        results_grid[(k_sig, h_sig)] = {**prf, **lead}

# Plot recall vs FAR tradeoff
fig, ax = plt.subplots(figsize=(7, 5))
colors = plt.cm.viridis(np.linspace(0, 0.9, len(h_sigmas)))
for ci, h_sig in enumerate(h_sigmas):
    recalls = [results_grid[(k, h_sig)]['recall_super'] for k in k_sigmas]
    fars    = [results_grid[(k, h_sig)]['far_sub']      for k in k_sigmas]
    ax.plot(recalls, fars, 'o-', color=colors[ci], lw=1.5, ms=6,
            label=f'h={h_sig}')
    for ki, k in enumerate(k_sigmas):
        ax.annotate(f'k={k}', (recalls[ki], fars[ki]),
                    textcoords='offset points', xytext=(5, 2), fontsize=7)

ax.set_xlabel('Recall (supercritical)')
ax.set_ylabel('FAR (subcritical false alarm rate)')
ax.set_title('Table 8 spirit: CUSUM operating-point sweep\n'
             'Higher k → lower recall, lower FAR (more conservative)')
ax.legend(title='h_sigma')
ax.grid(alpha=0.3)
# Mark paper operating points
ax.scatter([0.52], [0.76], marker='★', s=200, color='gold', zorder=5,
           label='Paper recall-oriented (k=0.5, h=4)')
ax.scatter([0.04], [0.07], marker='★', s=200, color='red', zorder=5,
           label='Paper precision-oriented (k=2.0, h=4)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('../results/table8_spirit_cusum_sweep.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: results/table8_spirit_cusum_sweep.png")
print("Paper Table 8: recall-oriented (k=0.5, h=4.0) → recall=0.52, FAR=0.76, lead=272")
print("             precision-oriented (k=2.0, h=4.0) → recall=0.04, FAR=0.07, lead=178")

In [ ]:
# === Proposition 1 illustration: CSAD ↔ κ̄_OR bridge ===
# Visualizes the mean-field scaling: CSAD_t ≈ σ_ξ * sqrt(2/π) * (1 - κ̄_OR)^0.5

kappa_range = np.linspace(-0.5, 0.9, 200)
sigma_xi = 0.02  # assumed return noise level

# Eq. 7: CSAD_t = sigma_xi * sqrt(2/pi) * (1 - kappa_bar_OR)^(1/2)
csad_theory = sigma_xi * np.sqrt(2 / np.pi) * np.sqrt(np.maximum(1 - kappa_range, 0))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(kappa_range, csad_theory, 'b-', lw=2, label='Eq. 7: $\\sigma_\\xi \\sqrt{2/\\pi}(1-\\bar{\\kappa}_{OR})^{1/2}$')
ax.axvline(0, color='k', ls='--', lw=0.8)
ax.axvline(0.3, color='green', ls='--', lw=1, label='$\\theta_{geom}=0.30$')
ax.annotate('Strong herding\n(high κ̄, low CSAD)', xy=(0.6, csad_theory[np.argmin(np.abs(kappa_range-0.6))]),
            xytext=(0.5, 0.025), arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=9, color='red')
ax.annotate('No herding\n(low κ̄, high CSAD)', xy=(-0.3, csad_theory[np.argmin(np.abs(kappa_range+0.3))]),
            xytext=(-0.5, 0.012), arrowprops=dict(arrowstyle='->', color='blue'),
            fontsize=9, color='blue')
ax.set_xlabel('$\\bar{\\kappa}_{OR}(t)$ (agent-graph curvature)')
ax.set_ylabel('$\\text{CSAD}_t$ (return dispersion)')
ax.set_title('Proposition 1: Mean-field bridge\n'
             '$\\text{CSAD}_t$ monotonically decreasing in $\\bar{\\kappa}_{OR}$ — consistent with $\\hat{\\gamma}_3 < 0$')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../results/prop1_csad_kappa_bridge.png', dpi=120, bbox_inches='tight')
plt.show()
print("Paper Proposition 1: CSAD_t = σ_ξ√(2/π)(1−κ̄_OR)^(1/2) + O(N^{-1/2})")
print("Empirical γ₃ = -0.0072 (CI [-0.00769, -0.00602]) — sign-consistent")

## Summary of Visualizations

| Figure | File | Paper Reference |
|--------|------|-----------------|
| κ̄_OR leads V_a(t) | `results/fig1_spirit_kappa_leads_event.png` | Figure 1 spirit |
| V_eff × κ̄ cross-correlation | `results/fig5a_spirit_veff_xcorr.png` | Figure 5a spirit |
| Edge density + V_eff | `results/fig2_spirit_edge_density.png` | Figure 2 spirit |
| CUSUM operating sweep | `results/table8_spirit_cusum_sweep.png` | Table 8 spirit |
| Proposition 1 bridge | `results/prop1_csad_kappa_bridge.png` | Proposition 1 |

**Note:** All figures use synthetic/proxy data. For exact paper figures, run the full pipeline
with N=66 agents, 400 CWS trajectories, and LP-Wasserstein ORC (requires POT).